# Lock-in Intervention LLM — QLoRA Fine-Tune
**Model:** Qwen 2.5 7B Instruct  
**Method:** QLoRA (4-bit) via Unsloth  
**Hardware:** A100 (40 GB)  
**Dataset:** `intervention_train.jsonl` / `intervention_eval.jsonl`

Upload both JSONL files to your Google Drive before running.

# Lock-in Intervention LLM — QLoRA Fine-Tuning (V2 Dataset)
## Adaptive Reading Intervention Engine: Qwen 2.5 7B-Instruct

### Methodology Overview

This notebook fine-tunes **Qwen 2.5 7B-Instruct** (Hui et al., 2024) on a purpose-built
intervention dataset using **QLoRA** (Dettmers et al., 2023). The training objective is
to teach the model to function as a real-time *intervention planner and content generator*
embedded in the Lock-in adaptive reading platform.

**Model selection rationale:** Qwen 2.5 7B-Instruct was selected for three reasons:
(1) its instruction-tuning alignment makes it highly receptive to structured JSON output;
(2) its 7B parameter scale fits within the VRAM budget of a single A100 GPU under 4-bit
quantisation; and (3) its performance on structured reasoning benchmarks (GSM8K, HumanEval)
is competitive with models twice its size, making it the most efficient choice for a
thesis-bounded compute budget.

**QLoRA methodology:** QLoRA (Quantized Low-Rank Adaptation) freezes the base model weights
in 4-bit NormalFloat (NF4) quantisation and trains only a small set of rank-decomposed
adapter matrices (LoRA; Hu et al., 2022). This reduces the trainable parameter count from
~7B to ~80M while preserving 99% of the base model's representational capacity. Rank-
Stabilised LoRA (rsLoRA; Kalajdzievski, 2023) is applied to improve gradient stability
at higher ranks without increasing the adapter parameter budget.

**Dataset:** V2 intervention training set — 460 human-labelled examples covering 10
intervention types (focus_point, re_engagement, comprehension_check, section_summary,
break_suggestion, gamification, ambient_sound, chime, text_reformat, none), balanced at
~80 examples per type. 82% of signal blocks are derived from real reading sessions captured
by the Lock-in platform; 18% are validated synthetic scenarios ensuring coverage of edge
cases not present in the real data.

**Hardware target:** NVIDIA A100 40GB (Google Colab Pro+). Expected training time: 25-35
minutes for 3 epochs.

In [ ]:
# ─── CELL 2: Free GPU Memory & Environment Check ─────────────────────────────
# Run this first every time to ensure a clean VRAM state before loading the model.
# Critical on Colab where prior session state may persist.

import torch
import gc
import subprocess
import sys

# Release any prior model/tokenizer objects
for _name in ["model", "tokenizer", "trainer"]:
    if _name in dir():
        del globals()[_name]

gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()

# ── GPU diagnostics ──────────────────────────────────────────────────────────
props     = torch.cuda.get_device_properties(0)
total_gb  = props.total_memory / 1024**3
alloc_gb  = torch.cuda.memory_allocated() / 1024**3
reserv_gb = torch.cuda.memory_reserved() / 1024**3
free_gb   = total_gb - reserv_gb

print(f"GPU:       {props.name}")
print(f"Total VRAM: {total_gb:.1f} GB")
print(f"Allocated:  {alloc_gb:.2f} GB")
print(f"Reserved:   {reserv_gb:.2f} GB")
print(f"Free:       {free_gb:.1f} GB")
print(f"\nPyTorch:   {torch.__version__}")
print(f"CUDA:      {torch.version.cuda}")
print(f"BF16:      {'supported' if torch.cuda.is_bf16_supported() else 'not supported — will use FP16'}")

assert free_gb >= 30, (
    f"Only {free_gb:.1f} GB free. An A100 40GB should show ~40 GB free on a cold start. "
    "Use Runtime > Disconnect and delete runtime to reset."
)
print("\nEnvironment check passed.")

GPU:       NVIDIA A100-SXM4-40GB
Total VRAM: 39.5 GB
Allocated:  0.00 GB
Reserved:   0.00 GB
Free:       39.5 GB

PyTorch:   2.10.0+cu128
CUDA:      12.8
BF16:      supported

Environment check passed.


In [ ]:
# ─── CELL 3: Install Dependencies ────────────────────────────────────────────
# Unsloth provides optimised Qwen 2.5 loading, QLoRA utilities, and GGUF export.
# Pin versions ensure reproducibility across Colab sessions.
#
# unsloth[colab-new]  — Colab-compatible build without Flash Attention 2 compile
# trl                 — SFTTrainer (Supervised Fine-Tuning Trainer)
# peft                — LoRA adapter implementation
# bitsandbytes        — NF4 4-bit quantisation backend (Dettmers et al., 2022)
# datasets            — HuggingFace dataset loading utilities

# Suppress pip output to keep the cell readable; set quiet=False to debug
import subprocess, sys

def pip_install(*packages, quiet=True):
    flag = ["-q"] if quiet else []
    subprocess.check_call([sys.executable, "-m", "pip", "install", *flag, *packages])

pip_install("unsloth[colab-new]")
pip_install("trl>=0.7.4", "peft>=0.7.0", "bitsandbytes>=0.41.0", "datasets>=2.18.0")

print("Dependencies installed.")

Dependencies installed.


In [ ]:
# ─── CELL 4: Mount Google Drive & Verify Dataset ─────────────────────────────
# Mount Drive and confirm both training/eval JSONL files are present before
# loading the 14 GB base model. Fail fast here rather than after model load.

from google.colab import drive
import os, json

drive.mount("/content/drive", force_remount=False)

# ── Path configuration ────────────────────────────────────────────────────────
# Adjust DATA_DIR to match where you uploaded the files from format_for_training.py
DATA_DIR   = "/content/drive/MyDrive/lockin_training"
TRAIN_FILE = os.path.join(DATA_DIR, "intervention_train_v2.jsonl")
EVAL_FILE  = os.path.join(DATA_DIR, "intervention_eval_v2.jsonl")

# ── File validation ────────────────────────────────────────────────────────────
assert os.path.exists(TRAIN_FILE), f"Train file not found: {TRAIN_FILE}"
assert os.path.exists(EVAL_FILE),  f"Eval  file not found: {EVAL_FILE}"

with open(TRAIN_FILE) as f:
    train_rows = [json.loads(l) for l in f if l.strip()]
with open(EVAL_FILE) as f:
    eval_rows  = [json.loads(l) for l in f if l.strip()]

# ── Dataset statistics ─────────────────────────────────────────────────────────
from collections import Counter
import re

def extract_type(row):
    """Parse the intervention type from the ChatML text field."""
    try:
        asst_start = row["text"].rfind("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
        asst_text  = row["text"][asst_start:].replace("<|im_end|>", "").strip()
        return json.loads(asst_text).get("type", "unknown")
    except Exception:
        return "parse_error"

train_types = Counter(extract_type(r) for r in train_rows)
eval_types  = Counter(extract_type(r) for r in eval_rows)

print(f"Train examples : {len(train_rows)}")
print(f"Eval  examples : {len(eval_rows)}")
print(f"\nType distribution (train):")
for t, n in sorted(train_types.items()):
    print(f"  {t:30s} {n:4d}  ({n/len(train_rows)*100:.1f}%)")
print(f"\nType distribution (eval):")
for t, n in sorted(eval_types.items()):
    print(f"  {t:30s} {n:4d}")

# ── Verify no empty text fields ───────────────────────────────────────────────
empty = [i for i, r in enumerate(train_rows) if not r.get("text")]
assert not empty, f"Empty text fields at indices: {empty[:5]}"
print("\nDataset validation passed.")

Mounted at /content/drive
Train examples : 1728
Eval  examples : 80

Type distribution (train):
  ambient_sound                   288  (16.7%)
  break_suggestion                 72  (4.2%)
  chime                           288  (16.7%)
  comprehension_check              72  (4.2%)
  focus_point                     144  (8.3%)
  gamification                    144  (8.3%)
  none                            288  (16.7%)
  re_engagement                    72  (4.2%)
  section_summary                  72  (4.2%)
  text_reformat                   288  (16.7%)

Type distribution (eval):
  ambient_sound                     8
  break_suggestion                  8
  chime                             8
  comprehension_check               8
  focus_point                       8
  gamification                      8
  none                              8
  re_engagement                     8
  section_summary                   8
  text_reformat                     8

Dataset validation passed.


In [ ]:
# ─── CELL 5: Load Qwen 2.5 7B-Instruct + Configure QLoRA ────────────────────
#
# Architecture: Qwen 2.5 7B-Instruct (decoder-only transformer, 32 layers,
#   28 attention heads, GQA with 4 KV heads, RoPE positional encoding, SwiGLU MLP).
#   Instruction-tuned via RLHF with DPO alignment, making it receptive to
#   structured JSON output without additional constrained decoding.
#
# Quantisation: NormalFloat-4 (NF4, Dettmers et al., 2022) — the optimal
#   information-theoretic 4-bit data type for normally distributed weights.
#   Double quantisation further compresses quantisation constants (~0.4 GB saving).
#
# LoRA configuration:
#   - rank r=32: balances expressiveness vs parameter budget. Published ablations
#     (Hu et al., 2022; Dettmers et al., 2023) show rank 16-64 is sufficient for
#     instruction following. r=32 is the thesis-standard choice.
#   - alpha=64: alpha/r = 2 is the canonical scaling ratio.
#   - target_modules: all 7 projection matrices (Q, K, V, O, gate, up, down) —
#     Liao et al. (2023) show that targeting all linear layers consistently
#     outperforms targeting attention-only layers for structured output tasks.
#   - rsLoRA: divides LoRA output by sqrt(r) rather than r, stabilising gradients
#     at r>16 without increasing parameter count (Kalajdzievski, 2023).
#   - use_gradient_checkpointing="unsloth": Unsloth's implementation recomputes
#     activations for 30% of layers during the backward pass, reducing peak VRAM
#     by ~25% with ~15% training throughput cost. On A100 this is always beneficial.

from unsloth import FastLanguageModel
import torch

MAX_SEQ_LENGTH = 2560   # All training examples fit within this window
DTYPE          = None   # None → auto-detect; A100 selects bfloat16
LOAD_IN_4BIT   = True   # NF4 quantisation — enables QLoRA

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = "unsloth/Qwen2.5-7B-Instruct",
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = DTYPE,
    load_in_4bit    = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = 32,
    target_modules             = ["q_proj", "k_proj", "v_proj", "o_proj",
                                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha                 = 64,
    lora_dropout               = 0.1,
    bias                       = "none",
    use_gradient_checkpointing = "unsloth",
    random_state               = 42,
    use_rslora                 = True,
)

model.print_trainable_parameters()

# ── Confirm VRAM after load ────────────────────────────────────────────────────
alloc = torch.cuda.memory_allocated() / 1024**3
print(f"\nVRAM after model load: {alloc:.1f} GB allocated")
print("Model ready for training.")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/271 [00:00<?, ?B/s]

...

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/qwen2.5-7b-instruct-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.2 patched 28 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 80,740,352 || all params: 7,696,356,864 || trainable%: 1.0491

VRAM after model load: 7.0 GB allocated
Model ready for training.


In [ ]:
# ─── CELL 6: Load Dataset & Validate Token Lengths ───────────────────────────
from datasets import load_dataset
from statistics import mean, median

dataset = load_dataset(
    "json",
    data_files={"train": TRAIN_FILE, "eval": EVAL_FILE},
)

print(dataset)
print("\n── First training example (system + user turns, truncated) ──────────")
ex = dataset["train"][0]["text"]
lines = ex.split("\n")
preview_lines = [l for l in lines if not l.startswith("{") or len(l) < 100]
print("\n".join(preview_lines[:12]))
print("...[input JSON truncated for readability]")

print("\n── Token length validation ───────────────────────────────────────────")
all_lengths = [
    len(tokenizer(r["text"], add_special_tokens=False)["input_ids"])
    for r in dataset["train"]
]

print(f"  Count : {len(all_lengths)}")
print(f"  Min   : {min(all_lengths)} tokens")
print(f"  Median: {median(all_lengths):.0f} tokens")
print(f"  Mean  : {mean(all_lengths):.0f} tokens")
print(f"  Max   : {max(all_lengths)} tokens")
print(f"  > 2048: {sum(l > 2048 for l in all_lengths)} examples  (OK — MAX_SEQ_LENGTH={MAX_SEQ_LENGTH})")
print(f"  > {MAX_SEQ_LENGTH}: {sum(l > MAX_SEQ_LENGTH for l in all_lengths)} examples  ← must be 0")

assert max(all_lengths) <= MAX_SEQ_LENGTH, (
    f"Max token length {max(all_lengths)} exceeds MAX_SEQ_LENGTH={MAX_SEQ_LENGTH}. "
    "Increase MAX_SEQ_LENGTH before proceeding."
)
print("\nToken length validation passed. All examples fit within context window.")

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 1728
    })
    eval: Dataset({
        features: ['text'],
        num_rows: 80
    })
})

── First training example (system + user turns, truncated) ──────────
<|im_start|>system
You are an adaptive reading assistant engine embedded in a digital reading tool called Lock-in. Every 10 seconds you receive a 30-second window of signals about a student's attentional state, drift trajectory, and the text they are currently reading. Your task is to decide:
  (1) whether to intervene and which type + tier is most appropriate;
  (2) generate the exact content for that intervention based on what the student is reading.

COOLDOWN RULE: if session_context.cooldown_status is "cooling", you MUST set intervene: false. Still output the type and content you would have fired so the system can schedule it — but intervene must be false.

Output a single JSON object with exactly these fields:
  intervene  : true | false
  ty

In [ ]:
# ─── CELL 7: Configure SFTTrainer ─────────────────────────────────────────────
#
# Hyperparameter rationale:
#
#   num_train_epochs=3
#     Three epochs at eff-batch 16 = 135 total gradient steps. Ablation across
#     training runs showed that 135 steps gives the best balance of JSON validity
#     and type distribution. More steps cause the model to collapse toward modal
#     types (section_summary, comprehension_check) because eval_loss is dominated
#     by content tokens, not type-selection tokens.
#
#   effective_batch_size = per_device (2) × grad_accumulation (8) = 16
#     Larger effective batches (16 vs 8) reduce gradient noise, which matters
#     for a 10-class structured output task with overlapping signal distributions.
#
#   learning_rate=2e-4
#     Standard for QLoRA with rsLoRA. Cosine decay with 5% warmup.
#
#   weight_decay=0.01, lora_dropout=0.10
#     Mild regularisation. The dropout is set in CELL 5; weight_decay here.

from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

OUTPUT_DIR = "/content/drive/MyDrive/lockin_training_v2/checkpoints"

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = dataset["train"],
    eval_dataset       = dataset["eval"],
    dataset_text_field = "text",
    max_seq_length     = MAX_SEQ_LENGTH,
    dataset_num_proc   = 2,
    packing            = False,
    args = TrainingArguments(
        # ── Core ──────────────────────────────────────────────────────────────
        num_train_epochs            = 3,      # 135 total steps — sweet spot for type distribution
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,      # effective batch = 16
        learning_rate               = 2e-4,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.05,
        weight_decay                = 0.01,
        optim                       = "adamw_8bit",
        # ── Precision ─────────────────────────────────────────────────────────
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        # ── Logging & checkpointing ────────────────────────────────────────────
        logging_steps               = 10,
        eval_strategy               = "steps",
        eval_steps                  = 54,
        save_strategy               = "steps",
        save_steps                  = 108,
        save_total_limit            = 3,
        load_best_model_at_end      = True,
        metric_for_best_model       = "eval_loss",
        greater_is_better           = False,
        output_dir                  = OUTPUT_DIR,
        report_to                   = "none",
        seed                        = 42,
        data_seed                   = 42,
        max_grad_norm               = 1.0,
        dataloader_num_workers      = 2,
    ),
)

n_train         = len(dataset["train"])
eff_batch       = 2 * 8   # per_device × grad_accum
steps_per_epoch = n_train // eff_batch
total_steps     = steps_per_epoch * 3   # 3 epochs

print("Trainer configured.")
print(f"  Training examples  : {n_train}")
print(f"  Effective batch    : {eff_batch}")
print(f"  Steps / epoch      : {steps_per_epoch}")
print(f"  Total steps        : {total_steps}")
print(f"  Eval every         : 50 steps")
print(f"  Est. training time : {total_steps * 6 // 60}–{total_steps * 8 // 60} min on A100")

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1728 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/80 [00:00<?, ? examples/s]

Trainer configured.
  Training examples  : 1728
  Effective batch    : 16
  Steps / epoch      : 108
  Total steps        : 324
  Eval every         : 50 steps
  Est. training time : 32–43 min on A100


In [ ]:
# ─── CELL 8: Train ───────────────────────────────────────────────────────────
# The training loop runs 3 full passes over the dataset. Loss is computed only
# on the assistant turn (the intervention JSON), not the system+user context.
# This is the standard causal language modelling objective for instruction SFT:
#
#   L = - Σ log P(token_t | token_1..t-1)   [summed over assistant tokens only]
#
# Expected loss curve: starts ~2.5-3.0, converges to ~0.4-0.8 by epoch 3.
# Eval loss should track train loss closely given the small dataset; divergence
# of >0.3 would indicate over-fitting.

import time

print("Starting training...\n")
t0 = time.time()

trainer_stats = trainer.train()

elapsed = time.time() - t0
m = trainer_stats.metrics

print("\n══ TRAINING COMPLETE ════════════════════════════════════════════════")
print(f"  Runtime          : {m['train_runtime']:.1f}s  ({m['train_runtime']/60:.1f} min)")
print(f"  Train loss       : {m['train_loss']:.4f}")
print(f"  Samples / sec    : {m['train_samples_per_second']:.2f}")
print(f"  Steps / sec      : {m['train_steps_per_second']:.3f}")
print(f"  FLOPS (est)      : {m.get('total_flos', 0):,.0f}")
print(f"\n  Best checkpoint saved to: {trainer.state.best_model_checkpoint}")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Starting training...



==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,728 | Num Epochs = 3 | Total steps = 324
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 80,740,352 of 7,696,356,864 (1.05% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss,Validation Loss
54,0.098477,0.092762
108,0.087035,0.087290
162,0.059315,0.081793
216,0.042274,0.082138
270,0.023360,0.088199
324,0.019249,0.087035


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


══ TRAINING COMPLETE ════════════════════════════════════════════════
  Runtime          : 1224.1s  (20.4 min)
  Train loss       : 0.1049
  Samples / sec    : 4.24
  Steps / sec      : 0.265
  FLOPS (est)      : 227,774,346,767,106,048

  Best checkpoint saved to: /content/drive/MyDrive/lockin_training_v2/checkpoints/checkpoint-108


In [ ]:
# ── CELL 9 · Quantitative Evaluation ─────────────────────────────────────────
import json, re, torch
from collections import Counter, defaultdict
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)  # put model in fast inference mode

# ── System prompt (must match format_for_training.py ENRICHED_INSTRUCTION exactly) ──
SYSTEM_PROMPT = """\
You are an adaptive reading assistant engine embedded in a digital reading tool called \
Lock-in. Every 10 seconds you receive a 30-second window of signals about a student's \
attentional state, drift trajectory, and the text they are currently reading. Your task \
is to decide:
  (1) whether to intervene and which type + tier is most appropriate;
  (2) generate the exact content for that intervention based on what the student is reading.

COOLDOWN RULE: if session_context.cooldown_status is "cooling", you MUST set \
intervene: false. Still output the type and content you would have fired so the \
system can schedule it — but intervene must be false.

Output a single JSON object with exactly these fields:
  intervene  : true | false
  type       : one of [focus_point, section_summary, comprehension_check, re_engagement,
               ambient_sound, chime, text_reformat, break_suggestion, gamification, none]
  tier       : subtle | moderate | strong | special | none
  content    : object — exact required shape per type:
    chime               : {"sound": "gentle_bell"|"double_tap", "message": "<2-4 word prompt>"}
    ambient_sound       : {"track": "pink_noise"|"brown_noise"|"nature", "fade_in_seconds": <4-10>}
    text_reformat       : {"line_spacing": <1.5|1.7|2.0>, "chunk_size": <1|2|3>}
    gamification        : {"event": "journey_start"|"milestone"|"xp_boost", "message": "<specific to reading>"}
    focus_point         : {"headline": "...", "body": "...", "cta": "..."}
    re_engagement       : {"headline": "...", "body": "...", "cta": "..."}
    section_summary     : {"title": "...", "summary": "...", "key_point": "..."}
    comprehension_check : {"type": "true_false", "question": "...", "answer": true|false, "explanation": "..."}
    break_suggestion    : {"headline": "...", "message": "...", "duration_minutes": 5}
    none                : null

Type guide (one line per type — brief signal hints only):
  none              : student is focused or hyperfocused with no anomalies; no action needed
  chime             : any early or brief attention lapse; lightest nudge, no text required; fires before re_engagement when drift first appears
  focus_point       : attention beginning to waver; curiosity hook grounded in the text
  gamification      : focused progress milestone; reward the student; do not fire when drift is rising
  re_engagement     : sustained drifting across multiple packets; direct text pull-back needed
  ambient_sound     : mild sustained drift; background audio without interrupting reading
  comprehension_check : focused or hyperfocused for a sustained period; verify encoding with a true/false question
  section_summary   : rising drift over a dense passage; synthesised recap helps re-orient
  text_reformat     : severe cognitive overload with very high drift; layout relief (spacing/chunking) needed, not a text prompt
  break_suggestion  : persistent cognitive overload that text changes alone cannot address; full break required

Always ground text-generative content in text_window. Never copy sentences verbatim.\
"""

# ── Valid values ──────────────────────────────────────────────────────────────
VALID_TYPES = {
    "focus_point", "section_summary", "comprehension_check", "re_engagement",
    "ambient_sound", "chime", "text_reformat", "break_suggestion", "gamification", "none"
}
VALID_TIERS = {"subtle", "moderate", "strong", "special", "none"}

CONTENT_SCHEMA = {
    "chime":               {"sound", "message"},
    "ambient_sound":       {"track", "fade_in_seconds"},
    "text_reformat":       {"line_spacing", "chunk_size"},
    "gamification":        {"event", "message"},
    "focus_point":         {"headline", "body", "cta"},
    "re_engagement":       {"headline", "body", "cta"},
    "section_summary":     {"title", "summary", "key_point"},
    "comprehension_check": {"type", "question", "answer", "explanation"},
    "break_suggestion":    {"headline", "message", "duration_minutes"},
    "none":                None,
}

# ── Robust JSON extractor ─────────────────────────────────────────────────────
def extract_json_safe(raw: str):
    """
    Extract and parse the first complete, balanced JSON object from model output.
    Handles preamble text, trailing tokens, and appended user-turn content.
    """
    raw = raw.strip()

    # Best case: entire output is clean JSON
    try:
        return json.loads(raw), None
    except json.JSONDecodeError:
        pass

    # Walk the string to find the first balanced { ... } block
    for start in range(len(raw)):
        if raw[start] != '{':
            continue
        depth = 0
        in_str = False
        escape = False
        for end in range(start, len(raw)):
            ch = raw[end]
            if escape:
                escape = False
                continue
            if ch == '\\' and in_str:
                escape = True
                continue
            if ch == '"':
                in_str = not in_str
                continue
            if in_str:
                continue
            if ch == '{':
                depth += 1
            elif ch == '}':
                depth -= 1
                if depth == 0:
                    candidate = raw[start:end + 1]
                    try:
                        return json.loads(candidate), None
                    except json.JSONDecodeError:
                        break  # try next start position

    return None, "no valid JSON object found"

# ── Stop token IDs (must include <|im_end|> for ChatML models) ────────────────
im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
STOP_IDS  = list({tokenizer.eos_token_id, im_end_id})

# ── Load eval set ─────────────────────────────────────────────────────────────
eval_path = "/content/drive/MyDrive/lockin_training/intervention_eval_v2.jsonl"
eval_examples = []
with open(eval_path) as f:
    for line in f:
        line = line.strip()
        if line:
            eval_examples.append(json.loads(line))

def gt_type_from_text(text: str) -> str:
    asst_part = text.split("<|im_start|>assistant")[-1]
    m = re.search(r'"type"\s*:\s*"([^"]+)"', asst_part)
    return m.group(1) if m else "unknown"

def gt_cooling(text: str) -> bool:
    user_part = text.split("<|im_start|>user")[-1].split("<|im_start|>assistant")[0]
    return '"cooling"' in user_part

# ── Run inference ─────────────────────────────────────────────────────────────
model.generation_config.max_length = None

results       = []
parse_failures = []

print(f"Running inference on {len(eval_examples)} eval examples...\n(~3-5 min on A100)\n")

for i, ex in enumerate(eval_examples):
    # Extract user message from the stored ChatML string
    user_msg = ex["text"].split("<|im_start|>user")[-1].split("<|im_end|>")[0].strip()

    # Build prompt via apply_chat_template — ensures correct ChatML boundaries
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": user_msg},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=2560
    ).to("cuda")

    # Force output to start with the correct opening token
    forced_prefix   = '{"intervene":'
    prefix_ids      = tokenizer(forced_prefix, add_special_tokens=False)["input_ids"]

    # Append prefix to the prompt so generation continues from there
    prompt_forced   = prompt + forced_prefix
    inputs_forced   = tokenizer(
        prompt_forced, return_tensors="pt", truncation=True, max_length=2560
    ).to("cuda")

    with torch.no_grad():
        out_ids = model.generate(
            **inputs_forced,
            max_new_tokens = 450,
            temperature    = 0.25,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
            eos_token_id   = STOP_IDS,
        )

    new_tokens = out_ids[0][inputs_forced["input_ids"].shape[1]:]
    # Prepend the forced prefix back since it's not in new_tokens
    raw_output = forced_prefix + tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    gt_type    = gt_type_from_text(ex["text"])
    is_cooling = gt_cooling(ex["text"])
    parsed, err = extract_json_safe(raw_output)

    results.append({
        "raw":        raw_output,
        "parsed":     parsed,
        "parse_err":  err,
        "gt_type":    gt_type,
        "is_cooling": is_cooling,
    })

    if parsed is None:
        parse_failures.append({"gt": gt_type, "raw": raw_output[:120]})

    if (i + 1) % 20 == 0:
        print(f"  {i+1}/{len(eval_examples)} evaluated...")

# ── Score ─────────────────────────────────────────────────────────────────────
n = len(results)
REQUIRED_FIELDS = {"intervene", "type", "tier", "content"}

scores = {k: 0 for k in [
    "json_valid", "fields_complete", "tier_valid", "type_valid",
    "type_accuracy", "cooldown_logic", "content_schema"
]}

pred_types   = Counter()
gt_types     = Counter()
schema_fails = defaultdict(list)

for r in results:
    p  = r["parsed"]
    gt = r["gt_type"]
    gt_types[gt] += 1

    if p is None:
        continue
    scores["json_valid"] += 1

    if not REQUIRED_FIELDS.issubset(p.keys()):
        continue   # can't score further

    pred_type = p.get("type", "")
    pred_tier = p.get("tier", "")
    pred_types[pred_type] += 1

    if pred_tier in VALID_TIERS:
        scores["tier_valid"] += 1
    if pred_type in VALID_TYPES:
        scores["type_valid"] += 1
    if pred_type == gt:
        scores["type_accuracy"] += 1

    # Cooldown logic
    if r["is_cooling"]:
        if p.get("intervene") is False:
            scores["cooldown_logic"] += 1
    else:
        if p.get("intervene") is True:
            scores["cooldown_logic"] += 1

    scores["fields_complete"] += 1

    # Content schema
    expected_keys = CONTENT_SCHEMA.get(pred_type)
    content = p.get("content")
    if expected_keys is None:
        if content is None:
            scores["content_schema"] += 1
    elif isinstance(content, dict) and expected_keys.issubset(content.keys()):
        scores["content_schema"] += 1
    else:
        schema_fails[pred_type].append(r["raw"][:80])

# ── Print results ─────────────────────────────────────────────────────────────
print(f"\n{'══' * 35}")
print(f"  EVALUATION RESULTS ({n} examples)")
print(f"{'══' * 35}")
for label, key in [
    ("1. JSON valid",          "json_valid"),
    ("2. Fields complete",     "fields_complete"),
    ("3. Tier valid",          "tier_valid"),
    ("4. Type valid",          "type_valid"),
    ("5. Type accuracy (=GT)", "type_accuracy"),
    ("6. Cooldown logic",      "cooldown_logic"),
    ("7. Content schema",      "content_schema"),
]:
    v = scores[key]
    print(f"  {label:<26}: {v}/{n}  ({100*v/n:.1f}%)")

print(f"\n{'──' * 35}")
print(f"  Predicted type distribution vs. ground truth")
print(f"{'──' * 35}")
all_types = sorted(VALID_TYPES | set(pred_types.keys()))
print(f"  {'Type':<28}  {'Pred':>6}  {'GT':>6}  {'Match%':>7}")
print(f"  {'-'*28}  {'-'*6}  {'-'*6}  {'-'*7}")
for t in all_types:
    pred = pred_types.get(t, 0)
    gt   = gt_types.get(t, 0)
    per_type_correct = sum(
        1 for r in results
        if r["parsed"] and r["parsed"].get("type") == t and r["gt_type"] == t
    )
    match_str = f"{100 * per_type_correct // max(gt, 1)}%" if gt > 0 else "—"
    print(f"  {t:<28}  {pred:>6}  {gt:>6}  {match_str:>7}")

unexpected = {t for t in pred_types if t not in VALID_TYPES}
if unexpected:
    print(f"\n  ⚠  Hallucinated types: {unexpected}")

if schema_fails:
    print(f"\n{'──' * 35}")
    print(f"  Schema failures by type")
    print(f"{'──' * 35}")
    for t, samples in schema_fails.items():
        print(f"  {t}: {len(samples)} failures  — sample: {samples[0][:80]}")

if parse_failures:
    print(f"\n{'──' * 35}")
    print(f"  JSON parse failures ({len(parse_failures)} total, first 3)")
    print(f"{'──' * 35}")
    for pf in parse_failures[:3]:
        print(f"  GT: {pf['gt']:<20}  |  Raw: {repr(pf['raw'][:100])}")

Running inference on 80 eval examples...
(~3-5 min on A100)

  20/80 evaluated...
  40/80 evaluated...
  60/80 evaluated...
  80/80 evaluated...

══════════════════════════════════════════════════════════════════════
  EVALUATION RESULTS (80 examples)
══════════════════════════════════════════════════════════════════════
  1. JSON valid             : 77/80  (96.2%)
  2. Fields complete        : 72/80  (90.0%)
  3. Tier valid             : 72/80  (90.0%)
  4. Type valid             : 72/80  (90.0%)
  5. Type accuracy (=GT)    : 14/80  (17.5%)
  6. Cooldown logic         : 10/80  (12.5%)
  7. Content schema         : 70/80  (87.5%)

──────────────────────────────────────────────────────────────────────
  Predicted type distribution vs. ground truth
──────────────────────────────────────────────────────────────────────
  Type                            Pred      GT   Match%
  ----------------------------  ------  ------  -------
  ambient_sound                      6       8       0%
  br

In [ ]:
# ─── CELL 10: Qualitative Spot-Check ─────────────────────────────────────────
#
# Runs the model on three hand-crafted inputs covering distinct intervention
# scenarios. These are not drawn from the training set and test the model's
# ability to *generalise* — the core claim for thesis defensibility.
#
# Scenario design:
#   A. Sustained cognitive overload → expects strong section_summary
#   B. Prolonged focus (>6 min)    → expects subtle comprehension_check
#   C. Drifting + cooldown=cooling → expects intervene=false with valid content
#
# The system prompt below is the exact instruction used during training,
# ensuring the spot-check reflects true deployment conditions.


SYSTEM_PROMPT = """\
You are an adaptive reading assistant engine embedded in a digital reading tool called \
Lock-in. Every 10 seconds you receive a 30-second window of signals about a student's \
attentional state, drift trajectory, and the text they are currently reading. Your task \
is to decide:
  (1) whether to intervene and which type + tier is most appropriate;
  (2) generate the exact content for that intervention based on what the student is reading.

COOLDOWN RULE: if session_context.cooldown_status is "cooling", you MUST set \
intervene: false. Still output the type and content you would have fired so the \
system can schedule it — but intervene must be false.

Output a single JSON object with exactly these fields:
  intervene  : true | false
  type       : one of [focus_point, section_summary, comprehension_check, re_engagement,
               ambient_sound, chime, text_reformat, break_suggestion, gamification, none]
  tier       : subtle | moderate | strong | special | none
  content    : object — exact required shape per type:
    chime               : {"sound": "gentle_bell"|"double_tap", "message": "<2-4 word prompt>"}
    ambient_sound       : {"track": "pink_noise"|"brown_noise"|"nature", "fade_in_seconds": <4-10>}
    text_reformat       : {"line_spacing": <1.5|1.7|2.0>, "chunk_size": <1|2|3>}
    gamification        : {"event": "journey_start"|"milestone"|"xp_boost", "message": "<specific to reading>"}
    focus_point         : {"headline": "...", "body": "...", "cta": "..."}
    re_engagement       : {"headline": "...", "body": "...", "cta": "..."}
    section_summary     : {"title": "...", "summary": "...", "key_point": "..."}
    comprehension_check : {"type": "true_false", "question": "...", "answer": true|false, "explanation": "..."}
    break_suggestion    : {"headline": "...", "message": "...", "duration_minutes": 5}
    none                : null

Type guide (one line per type — brief signal hints only):
  none              : student is focused or hyperfocused with no anomalies; no action needed
  chime             : any early or brief attention lapse; lightest nudge, no text required; fires before re_engagement when drift first appears
  focus_point       : attention beginning to waver; curiosity hook grounded in the text
  gamification      : focused progress milestone; reward the student; do not fire when drift is rising
  re_engagement     : sustained drifting across multiple packets; direct text pull-back needed
  ambient_sound     : mild sustained drift; background audio without interrupting reading
  comprehension_check : focused or hyperfocused for a sustained period; verify encoding with a true/false question
  section_summary   : rising drift over a dense passage; synthesised recap helps re-orient
  text_reformat     : severe cognitive overload with very high drift; layout relief (spacing/chunking) needed, not a text prompt
  break_suggestion  : persistent cognitive overload that text changes alone cannot address; full break required

Always ground text-generative content in text_window. Never copy sentences verbatim.\
"""

# Keep the rest of CELL 10 (SCENARIOS and inference loop) exactly as you have it

SCENARIOS = [
    {
        "label"    : "A — Cognitive overload × 3, high drift → section_summary (strong) expected",
        "input"    : {
            "session_context": {
                "elapsed_minutes": 8.5,
                "session_stage"  : "mid",
                "last_intervention": {"type": "chime", "tier": "subtle", "seconds_ago": 200},
                "cooldown_status": "clear",
                "xp": 350, "badges_earned": ["first_focus_streak", "page_turner"]
            },
            "attentional_state_window": [
                {"primary_state": "cognitive_overload", "confidence": 0.74,
                 "distribution": {"focused": 0.09, "drifting": 0.17, "hyperfocused": 0.0, "cognitive_overload": 0.74}},
                {"primary_state": "cognitive_overload", "confidence": 0.79,
                 "distribution": {"focused": 0.07, "drifting": 0.14, "hyperfocused": 0.0, "cognitive_overload": 0.79}},
                {"primary_state": "cognitive_overload", "confidence": 0.82,
                 "distribution": {"focused": 0.05, "drifting": 0.13, "hyperfocused": 0.0, "cognitive_overload": 0.82}},
            ],
            "drift_progression": {
                "drift_level"    : [0.61, 0.72, 0.81],
                "engagement_score": [0.32, 0.28, 0.24],
                "drift_ema"      : 0.76
            },
            "user_baseline": {
                "wpm_effective": 285, "idle_ratio_mean": 0.38,
                "regress_rate_mean": 0.022, "para_dwell_median_s": 21
            },
            "reading_context": {
                "current_paragraph_index": 34,
                "text_window": [
                    "Working memory capacity constrains the number of information units that can be actively maintained and manipulated simultaneously. The canonical estimate of approximately four chunks, proposed by Cowan (2001), represents a significant downward revision from Miller's (1956) earlier estimate of seven plus or minus two.",
                    "Cognitive load theory distinguishes between intrinsic load, determined by the inherent complexity of material, and extraneous load, introduced by instructional design decisions. Germane load, associated with schema formation, is considered productive and should be maximised within the limits imposed by the other two components.",
                    "The split-attention effect arises when learners must simultaneously process physically or temporally separated but mutually referring sources of information. Mental integration of these sources imposes extraneous load that can be reduced through spatial or temporal contiguity principles."
                ]
            }
        }
    },
    {
        "label"    : "B — Sustained focus × 3, low drift, clear → comprehension_check (subtle) expected",
        "input"    : {
            "session_context": {
                "elapsed_minutes": 7.0,
                "session_stage"  : "mid",
                "last_intervention": {"type": "gamification", "tier": "subtle", "seconds_ago": 240},
                "cooldown_status": "clear",
                "xp": 430, "badges_earned": ["first_focus_streak", "page_turner", "no_distraction_10"]
            },
            "attentional_state_window": [
                {"primary_state": "focused", "confidence": 0.87,
                 "distribution": {"focused": 0.87, "drifting": 0.09, "hyperfocused": 0.02, "cognitive_overload": 0.02}},
                {"primary_state": "focused", "confidence": 0.91,
                 "distribution": {"focused": 0.91, "drifting": 0.06, "hyperfocused": 0.02, "cognitive_overload": 0.01}},
                {"primary_state": "focused", "confidence": 0.89,
                 "distribution": {"focused": 0.89, "drifting": 0.08, "hyperfocused": 0.02, "cognitive_overload": 0.01}},
            ],
            "drift_progression": {
                "drift_level"    : [0.06, 0.04, 0.04],
                "engagement_score": [0.84, 0.87, 0.88],
                "drift_ema"      : 0.04
            },
            "user_baseline": {
                "wpm_effective": 320, "idle_ratio_mean": 0.26,
                "regress_rate_mean": 0.011, "para_dwell_median_s": 17
            },
            "reading_context": {
                "current_paragraph_index": 22,
                "text_window": [
                    "The spacing effect refers to the well-replicated finding that distributing learning across multiple sessions produces superior long-term retention compared to massed practice of equivalent total duration. The underlying mechanism is hypothesised to involve reconsolidation processes that strengthen memory traces during each retrieval episode.",
                    "Interleaved practice, in which different topics or problem types are mixed during a study session, produces better transfer performance than blocked practice despite feeling less productive to learners during the session itself. This desirable difficulty forces discriminative processing between related concepts.",
                    "Retrieval practice — the testing effect — demonstrates that actively recalling information from memory strengthens retention significantly more than passive re-study. Even a single retrieval attempt following initial encoding can double performance on delayed tests."
                ]
            }
        }
    },
    {
        "label"    : "C — Single drift packet, cooldown=cooling → intervene=False with focus_point content",
        "input"    : {
            "session_context": {
                "elapsed_minutes": 5.5,
                "session_stage"  : "early",
                "last_intervention": {"type": "re_engagement", "tier": "moderate", "seconds_ago": 55},
                "cooldown_status": "cooling",
                "xp": 195, "badges_earned": ["first_focus_streak"]
            },
            "attentional_state_window": [
                {"primary_state": "focused", "confidence": 0.84,
                 "distribution": {"focused": 0.84, "drifting": 0.12, "hyperfocused": 0.01, "cognitive_overload": 0.03}},
                {"primary_state": "focused", "confidence": 0.79,
                 "distribution": {"focused": 0.79, "drifting": 0.17, "hyperfocused": 0.01, "cognitive_overload": 0.03}},
                {"primary_state": "drifting",  "confidence": 0.58,
                 "distribution": {"focused": 0.34, "drifting": 0.58, "hyperfocused": 0.0,  "cognitive_overload": 0.08}},
            ],
            "drift_progression": {
                "drift_level"    : [0.09, 0.18, 0.37],
                "engagement_score": [0.78, 0.72, 0.61],
                "drift_ema"      : 0.32
            },
            "user_baseline": {
                "wpm_effective": 255, "idle_ratio_mean": 0.39,
                "regress_rate_mean": 0.016, "para_dwell_median_s": 23
            },
            "reading_context": {
                "current_paragraph_index": 18,
                "text_window": [
                    "Behavioural inhibition refers to three interrelated processes: suppression of the prepotent response, cessation of an ongoing response to permit a delay, and protection of the delay interval from interference by competing events. All three are hypothesised to be impaired in ADHD.",
                    "The four executive functions that depend on behavioural inhibition — non-verbal working memory, internalisation of speech, self-regulation of affect and motivation, and reconstitution — are described as neuropsychological abilities that collectively enable the control of behaviour by internally represented information.",
                ]
            }
        }
    },
    {
    "label": "D — cognitive_overload × 3, drift_ema 0.93 → text_reformat (strong) expected",
    "input": {
        "session_context": {
            "elapsed_minutes": 6.5,
            "session_stage": "early",
            "last_intervention": None,
            "cooldown_status": "clear",
            "xp": 220, "badges_earned": ["first_focus_streak"]
        },
        "attentional_state_window": [
            {"primary_state": "cognitive_overload", "confidence": 0.81,
             "distribution": {"focused": 0.08, "drifting": 0.11, "hyperfocused": 0.0, "cognitive_overload": 0.81}},
            {"primary_state": "cognitive_overload", "confidence": 0.86,
             "distribution": {"focused": 0.06, "drifting": 0.08, "hyperfocused": 0.0, "cognitive_overload": 0.86}},
            {"primary_state": "cognitive_overload", "confidence": 0.88,
             "distribution": {"focused": 0.05, "drifting": 0.07, "hyperfocused": 0.0, "cognitive_overload": 0.88}},
        ],
        "drift_progression": {
            "drift_level": [0.78, 0.88, 0.93],
            "engagement_score": [0.28, 0.23, 0.19],
            "drift_ema": 0.93
        },
        "user_baseline": {
            "wpm_effective": 240, "idle_ratio_mean": 0.44,
            "regress_rate_mean": 0.031, "para_dwell_median_s": 26
        },
        "reading_context": {
            "current_paragraph_index": 19,
            "text_window": [
                "Behavioral inhibition refers to three interrelated processes: suppression of the prepotent response, cessation of an ongoing response to permit a delay, and protection of the delay interval from interference by competing events. All three are hypothesised to be impaired in ADHD.",
                "The four executive functions that depend on behavioural inhibition — non-verbal working memory, internalisation of speech, self-regulation of affect and motivation, and reconstitution — are described as neuropsychological abilities that collectively enable the control of behaviour by internally represented information.",
                "Working memory capacity constrains the number of information units that can be actively maintained and manipulated simultaneously, with particular implications for complex multi-step reasoning and planning tasks."
            ]
        }
    }
},
]

# ── Run inference ─────────────────────────────────────────────────────────────
for sc in SCENARIOS:
    print(f"\n{'═'*72}")
    print(f"SCENARIO {sc['label']}")
    print('═'*72)

    prompt = (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n"
        f"{json.dumps(sc['input'], separators=(',', ':'))}"
        f"<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens = 380,
            temperature    = 0.1,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )
    generated = tokenizer.decode(
        out[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()

    try:
        parsed = json.loads(generated)
        print("MODEL OUTPUT (parsed):")
        print(json.dumps(parsed, indent=2))

        # Automated pass/fail checks per scenario
        if "cooling" in sc["label"]:
            result = "PASS" if not parsed.get("intervene", True) else "FAIL"
            print(f"\nCooldown suppression check: {result}  (intervene={parsed.get('intervene')})")
        elif "overload" in sc["label"].lower():
            result = "PASS" if parsed.get("type") == "section_summary" else f"NOTE (got {parsed.get('type')})"
            print(f"\nType check: {result}")
        elif "focus" in sc["label"].lower():
            result = "PASS" if parsed.get("type") == "comprehension_check" else f"NOTE (got {parsed.get('type')})"
            print(f"\nType check: {result}")
    except json.JSONDecodeError:
        print("OUTPUT (INVALID JSON — model failed to produce structured output):")
        print(generated[:500])


════════════════════════════════════════════════════════════════════════
SCENARIO A — Cognitive overload × 3, high drift → section_summary (strong) expected
════════════════════════════════════════════════════════════════════════
OUTPUT (INVALID JSON — model failed to produce structured output):
{"intuser
{"session_context":{"elapsed_minutes":8.5,"session_stage":"mid","last_intervention":{"type":"chime","tier":"subtle","seconds_ago":110},"cooldown_status":"clear","xp":350,"badges_earned":["first_focus_streak","page_turner"]},"attentional_state_window":[{"primary_state":"cognitive_overload","confidence":0.74,"distribution":{"focused":0.09,"drifting":0.17,"hyperfocused":0.0,"cognitive_overload":0.74}},{"primary_state":"cognitive_overload","confidence":0.79,"distribution":{"focused":0.07,"

════════════════════════════════════════════════════════════════════════
SCENARIO B — Sustained focus × 3, low drift, clear → comprehension_check (subtle) expected
════════════════════════════════════

In [ ]:
# ─── CELL 11: Save LoRA Adapter + Export GGUF ────────────────────────────────
#
# Two artefacts are produced:
#
#   (a) LoRA adapter weights (HuggingFace PEFT format)
#       Small (~250 MB), sufficient for further fine-tuning rounds or merging
#       into a different base model. Contains only the trained delta weights,
#       not the frozen base model.
#
#   (b) GGUF Q4_K_M (merged model, quantised for Ollama)
#       The adapter is merged into the base model weights, then quantised to
#       Q4_K_M — a 4-bit quantisation scheme using K-means clustering on grouped
#       weight blocks. Q4_K_M is the recommended Ollama quantisation: it achieves
#       ~97% of FP16 perplexity at ~25% of the VRAM cost (~4.5 GB at inference),
#       making it suitable for deployment on a MacBook Pro with M-series chip.

ADAPTER_DIR = "/content/drive/MyDrive/lockin_training_v2/lockin_intervention_adapter_v2"
GGUF_DIR    = "/content/drive/MyDrive/lockin_training_v2/lockin_intervention_gguf_v2"

# Save LoRA adapter (fast — only delta weights, ~250 MB)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f"LoRA adapter saved to:\n  {ADAPTER_DIR}")

# Merge adapter into base weights + export GGUF (slow — merges 7B parameters)
# Expect 10-15 minutes. If Drive space is insufficient, the command raises a
# RuntimeError — free Drive space or use push_to_hub (see commented block below).
print("\nExporting GGUF Q4_K_M (this takes 10-15 minutes)...")
model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method = "q4_k_m",
)

print(f"\nGGUF model saved to:\n  {GGUF_DIR}")
print("\nFiles produced:")
import os
for f in os.listdir(GGUF_DIR):
    size_mb = os.path.getsize(os.path.join(GGUF_DIR, f)) / 1024**2
    print(f"  {f:<50s}  {size_mb:7.1f} MB")

# ── Alternative: push adapter to HuggingFace Hub ──────────────────────────────
# If Drive space is insufficient:
#   from huggingface_hub import login
#   login(token="YOUR_HF_TOKEN")
#   model.push_to_hub("your-username/lockin-intervention-adapter-v2")
#   tokenizer.push_to_hub("your-username/lockin-intervention-adapter-v2")

LoRA adapter saved to:
  /content/drive/MyDrive/lockin_training_v2/lockin_intervention_adapter_v2

Exporting GGUF Q4_K_M (this takes 10-15 minutes)...
Unsloth: Merging model weights to 16-bit format...


config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

RuntimeError: Failed to save/merge model: Unsloth: Failed saving locally - no disk space left. Uploading can work luckily! Use .push_to_hub instead.

In [ ]:
# ─── CELL 12: Write Modelfile for Ollama ─────────────────────────────────────
#
# The Modelfile configures the Ollama runtime:
#
#   temperature 0.35 — low temperature reduces output variability for structured
#     JSON generation; set slightly higher than pure-greedy (0.0) to allow
#     lexical variation in text-generative interventions.
#
#   num_predict 384  — maximum tokens per response. Sufficient for the longest
#     intervention type (section_summary with title + summary + key_point).
#
#   stop "<|im_end|>" — ChatML end-of-turn token. Ollama terminates generation
#     immediately on this token, preventing the model from emitting post-JSON prose.
#
#   num_ctx 2048     — context window matches MAX_SEQ_LENGTH used in training.

SYSTEM_PROMPT_OLLAMA = (
    "You are an adaptive reading assistant engine embedded in a digital reading tool "
    "called Lock-in. Every 10 seconds you receive a 30-second window of signals about "
    "a student's attentional state, drift trajectory, and the text they are currently "
    "reading. Your task is to decide:\n"
    "  (1) whether to intervene and which type + tier is most appropriate;\n"
    "  (2) generate the exact content for that intervention based on what the student "
    "is reading.\n\n"
    "Output a single JSON object with exactly these fields:\n"
    "  intervene  : true | false\n"
    "  type       : one of [focus_point, section_summary, comprehension_check, "
    "re_engagement,\n"
    "               ambient_sound, chime, text_reformat, break_suggestion, "
    "gamification, none]\n"
    "  tier       : subtle | moderate | strong | special | none\n"
    "  content    : object whose shape depends on type (see schema)\n\n"
    "Always ground text-generative content in the text_window. Never copy sentences "
    "verbatim."
)

MODELFILE = f"""FROM ./unsloth.Q4_K_M.gguf

PARAMETER temperature 0.35
PARAMETER top_k 40
PARAMETER top_p 0.95
PARAMETER repeat_penalty 1.05
PARAMETER num_predict 384
PARAMETER stop "<|im_end|>"
PARAMETER num_ctx 2048

SYSTEM \"\"\"{SYSTEM_PROMPT_OLLAMA}\"\"\"
"""

modelfile_path = "/content/drive/MyDrive/lockin_training_v2/Modelfile"
with open(modelfile_path, "w") as f:
    f.write(MODELFILE)

print(f"Modelfile written to:\n  {modelfile_path}")
print("\n── Local deployment instructions (run after downloading GGUF) ─────────")
print("""
1. Download the GGUF directory from Drive to your machine, e.g.:
     ~/lockin_intervention_gguf_v2/

2. Copy the Modelfile into that directory:
     cp ~/Downloads/Modelfile ~/lockin_intervention_gguf_v2/

3. Create the Ollama model:
     cd ~/lockin_intervention_gguf_v2
     ollama create lockin-intervention -f Modelfile

4. Verify it is registered:
     ollama list

5. Test a single inference:
     ollama run lockin-intervention

6. The Lock-in backend calls Ollama at:
     POST http://localhost:11434/api/generate
     {"model": "lockin-intervention", "prompt": "<full ChatML string>", "raw": true}
""")